# 📈 Análisis Exploratorio de Datos — Mercado de Acciones (2020–2024)

**Autor:** Alejandro Gutierrez Viscay  
**Tecnologías:** Python · Pandas · NumPy · Matplotlib · Seaborn  
**Dataset:** Precios históricos de cierre de 8 acciones del S&P 500 (2020–2024)

---

## Objetivo

Explorar el comportamiento del mercado bursátil estadounidense durante el período 2020–2024, 
identificando patrones de rendimiento, correlaciones entre activos, volatilidad por sector y 
tendencias de largo plazo. Este análisis simula el tipo de trabajo analítico aplicado en 
contextos financieros y corporativos.

## Preguntas de análisis

1. ¿Qué acción tuvo el mejor rendimiento acumulado en el período?
2. ¿Qué sectores mostraron mayor volatilidad?
3. ¿Cómo se correlacionan los activos entre sí? ¿Existe diversificación real?
4. ¿Cuál fue el impacto del crash de COVID-19 en marzo 2020?
5. ¿Qué tan consistentes fueron los retornos anuales por activo?


## 1. Configuración e importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Estilo global
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2d3748',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#a0aec0',
    'ytick.color': '#a0aec0',
    'text.color': '#e2e8f0',
    'grid.color': '#2d3748',
    'grid.alpha': 0.5,
    'font.family': 'monospace',
    'figure.dpi': 120
})

PALETTE = ['#60a5fa', '#34d399', '#f59e0b', '#f87171', '#a78bfa', '#38bdf8', '#fb923c', '#e879f9']
TICKERS  = ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'XOM', 'JNJ', 'AMZN', 'BRK-B']
SECTORS  = {'AAPL':'Technology','MSFT':'Technology','GOOGL':'Technology',
            'JPM':'Finance','XOM':'Energy','JNJ':'Healthcare','AMZN':'Consumer','BRK-B':'Finance'}

print("✅ Librerías cargadas correctamente")
print(f"   pandas  {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}")


## 2. Carga y validación de datos

In [ ]:
df = pd.read_csv('stock_prices.csv', index_col='Date', parse_dates=True)
df_vol = pd.read_csv('stock_volume.csv', index_col='Date', parse_dates=True)

print("═" * 50)
print(f"  Período analizado: {df.index[0].date()} → {df.index[-1].date()}")
print(f"  Días de trading  : {len(df):,}")
print(f"  Activos          : {', '.join(df.columns)}")
print("═" * 50)
print()
print("Primeros registros:")
df.head()


## 3. Estadísticas descriptivas

In [ ]:
stats = df.describe().round(2)
print("Estadísticas de precios de cierre (USD):")
stats


In [ ]:
# Chequeo de valores nulos
nulls = df.isnull().sum()
print("Valores nulos por columna:")
print(nulls)
print(f"\n✅ Dataset limpio: {nulls.sum() == 0}")


## 4. Evolución histórica de precios

Visualizamos el precio de cierre ajustado de los 8 activos durante el período 2020–2024.
El crash de COVID-19 (marzo 2020) y la recuperación posterior son claramente visibles.


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 14))
fig.suptitle('Evolución de Precios de Cierre — S&P 500 (2020–2024)', 
             fontsize=16, fontweight='bold', y=1.01, color='#f8fafc')
axes = axes.flatten()

for i, (ticker, color) in enumerate(zip(TICKERS, PALETTE)):
    ax = axes[i]
    ax.plot(df.index, df[ticker], color=color, linewidth=1.5, alpha=0.9)
    ax.fill_between(df.index, df[ticker], alpha=0.1, color=color)
    ax.set_title(f'{ticker} — {SECTORS[ticker]}', fontsize=11, fontweight='bold', color=color)
    ax.set_ylabel('Precio (USD)', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.grid(True, alpha=0.3)
    
    # Marcar mínimo (crash COVID)
    min_idx = df[ticker].idxmin()
    ax.axvline(min_idx, color='#f87171', linestyle='--', alpha=0.5, linewidth=1)
    ax.annotate('COVID
crash', xy=(min_idx, df[ticker].min()),
                xytext=(10, 20), textcoords='offset points',
                fontsize=7, color='#f87171')

plt.tight_layout()
plt.savefig('assets/01_price_history.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("📊 Gráfico guardado.")


## 5. Rendimiento acumulado (2020–2024)

Normalizamos los precios al valor inicial para comparar el crecimiento proporcional de cada activo.
Un valor de 2.0 significa que el activo duplicó su valor desde el inicio del período.


In [ ]:
normalized = df / df.iloc[0] * 100  # Base 100

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_title('Rendimiento Acumulado Normalizado (Base 100 = Ene 2020)', 
             fontsize=14, fontweight='bold', pad=15)

for ticker, color in zip(TICKERS, PALETTE):
    ax.plot(normalized.index, normalized[ticker], label=ticker, 
            color=color, linewidth=2, alpha=0.85)

# Línea base
ax.axhline(100, color='#64748b', linestyle='--', alpha=0.5, linewidth=1, label='Inicio (100)')
ax.axvspan(pd.Timestamp('2020-02-20'), pd.Timestamp('2020-03-23'), 
           alpha=0.1, color='#f87171', label='Crash COVID-19')

ax.set_ylabel('Rendimiento (Base 100)', fontsize=11)
ax.legend(loc='upper left', fontsize=9, framealpha=0.2)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Anotar valor final de cada activo
for ticker, color in zip(TICKERS, PALETTE):
    final_val = normalized[ticker].iloc[-1]
    ax.annotate(f'{ticker}: {final_val:.0f}', 
                xy=(normalized.index[-1], final_val),
                xytext=(5, 0), textcoords='offset points',
                color=color, fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('assets/02_cumulative_returns.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()

# Ranking
final_returns = ((df.iloc[-1] / df.iloc[0] - 1) * 100).sort_values(ascending=False)
print("\n📊 Rendimiento total del período:")
for t, r in final_returns.items():
    bar = '█' * int(r / 50)
    print(f"  {t:6s} ({SECTORS[t]:12s})  {r:+6.1f}%  {bar}")


## 6. Retornos diarios y volatilidad

Los retornos diarios (variación porcentual día a día) nos permiten medir la volatilidad 
de cada activo. Una mayor dispersión implica mayor riesgo.


In [ ]:
daily_returns = df.pct_change().dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot de retornos
ax1 = axes[0]
bp_data = [daily_returns[t].values * 100 for t in TICKERS]
bp = ax1.boxplot(bp_data, labels=TICKERS, patch_artist=True, 
                 medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax1.axhline(0, color='#64748b', linestyle='--', alpha=0.5)
ax1.set_title('Distribución de Retornos Diarios', fontsize=12, fontweight='bold')
ax1.set_ylabel('Retorno Diario (%)', fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# Volatilidad anualizada
ann_vol = daily_returns.std() * np.sqrt(252) * 100
ax2 = axes[1]
bars = ax2.barh(TICKERS, ann_vol.values, color=PALETTE, alpha=0.8, edgecolor='none')
for bar, val in zip(bars, ann_vol.values):
    ax2.text(val + 0.3, bar.get_y() + bar.get_height()/2, 
             f'{val:.1f}%', va='center', fontsize=9, color='#e2e8f0')
ax2.set_title('Volatilidad Anualizada', fontsize=12, fontweight='bold')
ax2.set_xlabel('Volatilidad (%)', fontsize=10)
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('assets/03_volatility.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()

print("\n📊 Volatilidad anualizada (mayor = más riesgo):")
for t, v in ann_vol.sort_values(ascending=False).items():
    print(f"  {t:6s}  {v:.2f}%")


## 7. Correlación entre activos

La matriz de correlación muestra qué tan sincronizados se mueven los activos.  
Valores cercanos a **+1** = se mueven igual (baja diversificación)  
Valores cercanos a **0** = movimientos independientes (buena diversificación)


In [ ]:
corr = daily_returns.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True  # Solo triángulo inferior

cmap = sns.diverging_palette(10, 220, as_cmap=True)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=cmap,
            vmin=-1, vmax=1, center=0,
            linewidths=0.5, linecolor='#0f1117',
            annot_kws={'size': 11, 'weight': 'bold'},
            ax=ax, square=True, cbar_kws={'shrink': 0.8})

ax.set_title('Matriz de Correlación de Retornos Diarios', 
             fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('assets/04_correlation_matrix.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()

# Pares con mayor correlación
print("\n🔗 Pares más correlacionados:")
corr_pairs = corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
corr_stack = corr_pairs.stack().sort_values(ascending=False)
for (t1, t2), val in corr_stack.head(5).items():
    print(f"  {t1} ↔ {t2}: {val:.3f}")


## 8. Análisis por sector y año

Agrupamos el rendimiento anual por sector para identificar qué industrias dominaron 
en cada año del período analizado.


In [ ]:
# Retorno anual por activo
annual_returns = df.resample('YE').last().pct_change().dropna() * 100
annual_returns.index = annual_returns.index.year

# Heatmap de retornos anuales
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap
cmap2 = sns.diverging_palette(10, 130, as_cmap=True)
sns.heatmap(annual_returns.T, annot=True, fmt='.1f', cmap=cmap2,
            center=0, linewidths=0.5, linecolor='#0f1117',
            annot_kws={'size': 10, 'weight': 'bold'},
            ax=axes[0], cbar_kws={'label': 'Retorno (%)'})
axes[0].set_title('Retorno Anual por Activo (%)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Año', fontsize=10)

# Promedio por sector
sector_map = pd.Series(SECTORS)
sector_annual = annual_returns.T.copy()
sector_annual['Sector'] = sector_annual.index.map(SECTORS)
sector_avg = sector_annual.groupby('Sector').mean()

sector_avg.T.plot(kind='bar', ax=axes[1], color=sns.color_palette('Set2', len(sector_avg)),
                  alpha=0.85, edgecolor='none', width=0.7)
axes[1].set_title('Retorno Promedio Anual por Sector', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Retorno (%)', fontsize=10)
axes[1].set_xlabel('Año', fontsize=10)
axes[1].legend(fontsize=8, framealpha=0.2)
axes[1].axhline(0, color='#64748b', linestyle='--', alpha=0.5)
axes[1].tick_params(axis='x', rotation=0)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('assets/05_annual_returns.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()


## 9. Análisis del Crash COVID-19

El período febrero–abril 2020 representa uno de los crashes bursátiles más rápidos de la historia.
Medimos la caída máxima (*drawdown*) y el tiempo de recuperación por activo.


In [ ]:
pre_crash = df.loc[:'2020-02-19']
crash_bottom = df.loc['2020-02-20':'2020-04-30']
post_peak = df.loc['2020-02-20':]

# Drawdown máximo durante el crash
pre_price = df.loc['2020-02-19']
min_price = df.loc['2020-02-20':'2020-04-30'].min()
drawdown = ((min_price - pre_price) / pre_price * 100).sort_values()

# Días en recuperar precio pre-crash
recovery_days = {}
for t in TICKERS:
    peak = df.loc['2020-02-19', t]
    recovery = post_peak[post_peak[t] >= peak]
    recovery_days[t] = (recovery.index[0] - pd.Timestamp('2020-02-19')).days if len(recovery) > 0 else None

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Drawdown
colors_dd = ['#f87171' if v < -20 else '#fb923c' if v < -10 else '#fbbf24' for v in drawdown.values]
axes[0].barh(drawdown.index, drawdown.values, color=colors_dd, alpha=0.85, edgecolor='none')
for i, (idx, val) in enumerate(drawdown.items()):
    axes[0].text(val - 0.3, i, f'{val:.1f}%', va='center', ha='right', fontsize=9, color='white')
axes[0].axvline(0, color='#64748b', linewidth=0.8)
axes[0].set_title('Caída Máxima durante COVID-19 (%)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Drawdown (%)', fontsize=10)
axes[0].grid(True, alpha=0.3, axis='x')

# Días de recuperación
rec_df = pd.Series(recovery_days).dropna().sort_values()
colors_r = [PALETTE[TICKERS.index(t)] for t in rec_df.index]
axes[1].bar(rec_df.index, rec_df.values, color=colors_r, alpha=0.85, edgecolor='none')
for i, (idx, val) in enumerate(rec_df.items()):
    axes[1].text(i, val + 2, f'{int(val)}d', ha='center', fontsize=9, color='white')
axes[1].set_title('Días para Recuperar Precio Pre-Crash', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Días', fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('assets/06_covid_impact.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()


## 10. Conclusiones y hallazgos clave

### 🏆 Rendimiento
- El sector **tecnológico** (AAPL, MSFT, GOOGL, AMZN) dominó el período con los mayores rendimientos acumulados
- **AAPL** y **MSFT** mostraron el mejor balance entre rendimiento y volatilidad (Sharpe ratio implícito)
- **XOM** (Energía) fue el activo con menor rendimiento total, aunque con recuperación post-2022

### 📉 Volatilidad
- AMZN y AAPL presentaron la mayor volatilidad anualizada (~28-30%)
- JNJ (Healthcare) y BRK-B (Finance) son los activos más estables del portafolio
- La volatilidad en 2020 fue 2-3x mayor que años posteriores

### 🔗 Correlaciones
- Los activos tecnológicos (AAPL, MSFT, GOOGL, AMZN) están altamente correlacionados entre sí (>0.70)
- XOM (energía) muestra correlaciones bajas con el resto — buena opción de diversificación
- La correlación aumenta drásticamente en períodos de crisis (el crash COVID elevó correlaciones a >0.90)

### 🦠 Impacto COVID-19
- El crash de marzo 2020 generó caídas de -20% a -35% en menos de 5 semanas
- Los activos tecnológicos se recuperaron en 60-90 días, impulsados por la digitalización
- XOM tardó más de 400 días en recuperar su precio pre-crash

### 💡 Insights para portafolio
1. Diversificación sectorial real requiere incluir energía, healthcare y activos defensivos
2. Las correlaciones en crisis invalidan la diversificación aparente entre tecnológicas
3. El período 2020-2024 favoreció fuertemente a inversores con tolerancia al riesgo y horizonte largo


## Reproducibilidad

In [ ]:
import sys
print(f"Python : {sys.version.split()[0]}")
import pandas, numpy, matplotlib, seaborn
for lib in [pandas, numpy, matplotlib, seaborn]:
    print(f"{lib.__name__:12s}: {lib.__version__}")
